# Vérifier une affirmation et citer ses sources

Retrouver les bons articles ne suffit pas. Ici le système décide si une
affirmation est étayée ou contredite, et il indique les phrases exactes sur
lesquelles il s'appuie.

Deux modèles, comme dans le papier de Wadden. Le premier note chaque phrase des
articles retrouvés pour dire si elle justifie l'affirmation. Le second lit les
phrases retenues et choisit entre étayé, contredit et sans information.

Ce qu'il faut battre, sur le jeu de développement :

| | phrases | articles |
|---|---|---|
| zéro-shot entraîné sur FEVER, 2020 | 28,4 | 38,4 |
| VeriSci | 42,6 | 48,5 |
| VeriSci avec les bons articles fournis | 60,6 | 72,5 |

L'évaluation utilise le code d'AllenAI, pas le mien. Je l'ai vérifié en lui
donnant la vérité terrain comme prédiction : il rend bien 1,0 partout.

Le jeu de test a ses étiquettes cachées, donc tout se rapporte sur le jeu de
développement. Les réglages, eux, se font sur une part du jeu d'entraînement
mise de côté.

In [ ]:
!pip -q install transformers torch nltk pandas sentencepiece 2>&1 | tail -2
import json, os, random, re, subprocess, tarfile, time, urllib.request
from collections import Counter
import numpy as np
import torch

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
print(torch.cuda.get_device_name(0))

## Les données et l'évaluateur officiel

In [ ]:
if not os.path.exists("data/claims_dev.jsonl"):
    urllib.request.urlretrieve(
        "https://scifact.s3-us-west-2.amazonaws.com/release/latest/data.tar.gz", "scifact.tar.gz")
    tarfile.open("scifact.tar.gz").extractall(".")

os.makedirs("evaluate/lib", exist_ok=True)
base = "https://raw.githubusercontent.com/allenai/scifact/master/verisci/evaluate"
urllib.request.urlretrieve(f"{base}/pipeline.py", "evaluate/pipeline.py")
for fichier in ["__init__.py", "data.py", "metrics.py"]:
    urllib.request.urlretrieve(f"{base}/lib/{fichier}", f"evaluate/lib/{fichier}")

articles = {str(json.loads(l)["doc_id"]): json.loads(l)
            for l in open("data/corpus.jsonl", encoding="utf-8")}
tout_entrainement = [json.loads(l) for l in open("data/claims_train.jsonl", encoding="utf-8")]
developpement = [json.loads(l) for l in open("data/claims_dev.jsonl", encoding="utf-8")]

random.Random(0).shuffle(tout_entrainement)
coupure = int(0.15 * len(tout_entrainement))
reglage = tout_entrainement[:coupure]
entrainement = tout_entrainement[coupure:]

with open("data/claims_reglage.jsonl", "w") as f:
    for affirmation in reglage:
        f.write(json.dumps(affirmation) + "\n")

print(f"{len(articles)} articles")
print(f"{len(entrainement)} affirmations pour entraîner, {len(reglage)} pour régler, "
      f"{len(developpement)} pour rapporter")

## Retrouver les articles

Même BM25 que dans l'autre notebook. Je vérifie d'abord combien d'articles de
preuve il ramène : c'est ce qui plafonne tout le reste.

In [ ]:
from nltk.stem.porter import PorterStemmer

mots_vides = set('''a an and are as at be but by for if in into is it no not of on
or such that the their then there these they this to was will with'''.split())
racine = PorterStemmer()
deja_vu = {}

def decouper(texte):
    mots = []
    for mot in re.findall(r"[a-z0-9]+", texte.lower()):
        if mot in mots_vides:
            continue
        if mot not in deja_vu:
            deja_vu[mot] = racine.stem(mot)
        mots.append(deja_vu[mot])
    return mots


class BM25:
    def __init__(self, textes, k1=0.9, b=0.4):
        self.k1, self.b = k1, b
        decoupes = [decouper(t) for t in textes]
        self.nb_documents = len(decoupes)
        self.longueurs = np.array([len(d) for d in decoupes], dtype=np.float32)
        self.longueur_moyenne = self.longueurs.mean()
        occurrences = {}
        for i, mots in enumerate(decoupes):
            for mot, frequence in Counter(mots).items():
                occurrences.setdefault(mot, []).append((i, frequence))
        self.index = {}
        for mot, liste in occurrences.items():
            ou = np.array([x[0] for x in liste], dtype=np.int32)
            combien = np.array([x[1] for x in liste], dtype=np.float32)
            df = len(liste)
            self.index[mot] = (ou, combien,
                               np.log(1 + (self.nb_documents - df + 0.5) / (df + 0.5)))

    def noter(self, question):
        notes = np.zeros(self.nb_documents, dtype=np.float32)
        for mot in decouper(question):
            if mot not in self.index:
                continue
            ou, combien, rarete = self.index[mot]
            longueur = 1 - self.b + self.b * self.longueurs[ou] / self.longueur_moyenne
            notes[ou] += rarete * combien * (self.k1 + 1) / (combien + self.k1 * longueur)
        return notes


identifiants = list(articles)
bm25 = BM25([f"{articles[d]['title']} {' '.join(articles[d]['abstract'])}" for d in identifiants])

def retrouver(affirmations, combien=10):
    trouve = {}
    for affirmation in affirmations:
        notes = bm25.noter(affirmation["claim"])
        haut = np.argpartition(-notes, combien)[:combien]
        trouve[affirmation["id"]] = [identifiants[i] for i in haut[np.argsort(-notes[haut])]]
    return trouve


retrouves = {"reglage": retrouver(reglage), "developpement": retrouver(developpement)}

for combien in (3, 5, 10):
    bons = attendus = 0
    for affirmation in developpement:
        preuves = affirmation.get("evidence") or {}
        if not preuves:
            continue
        bons += len(set(preuves) & set(retrouves["developpement"][affirmation["id"]][:combien]))
        attendus += len(preuves)
    couverture = bons / attendus
    print(f"top-{combien:<3} {100*couverture:.1f} % des articles de preuve retrouvés, "
          f"soit un plafond de {100 * 2*couverture/(1+couverture):.1f} de F1")

## Entraînement, les outils

In [ ]:
from torch.utils.data import DataLoader, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)

def charger_modele(nom, nb_classes=None):
    options = {"num_labels": nb_classes} if nb_classes else {}
    modele = AutoModelForSequenceClassification.from_pretrained(nom, **options)
    return modele.float().to("cuda")


class Exemples(Dataset):
    def __init__(self, entrees, cibles):
        self.entrees, self.cibles = entrees, cibles
    def __len__(self):
        return len(self.entrees)
    def __getitem__(self, i):
        return self.entrees[i], int(self.cibles[i])


def entrainer(modele, tokenizer, entrees, cibles, epoques=3, taille_lot=32,
              pas=2e-5, longueur=256, poids_classes=None):
    def assembler(lot):
        paires = [x[0] for x in lot]
        y = torch.tensor([x[1] for x in lot])
        encode = tokenizer([a for a, _ in paires], [b for _, b in paires],
                           padding=True, truncation=True, max_length=longueur,
                           return_tensors="pt")
        return encode, y

    donnees = DataLoader(Exemples(entrees, cibles), batch_size=taille_lot,
                         shuffle=True, collate_fn=assembler)
    optimiseur = torch.optim.AdamW(modele.parameters(), lr=pas, weight_decay=0.01)
    etapes = len(donnees) * epoques
    calendrier = get_linear_schedule_with_warmup(optimiseur, int(0.1 * etapes), etapes)
    critere = torch.nn.CrossEntropyLoss(
        weight=poids_classes.to("cuda") if poids_classes is not None else None)
    echelle = torch.amp.GradScaler("cuda")

    modele.train()
    for epoque in range(epoques):
        cumul, depart = 0.0, time.time()
        for encode, cible in donnees:
            encode = {k: v.to("cuda") for k, v in encode.items()}
            cible = cible.to("cuda")
            optimiseur.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                perte = critere(modele(**encode).logits, cible)
            echelle.scale(perte).backward()
            echelle.unscale_(optimiseur)
            torch.nn.utils.clip_grad_norm_(modele.parameters(), 1.0)
            echelle.step(optimiseur)
            echelle.update()
            calendrier.step()
            cumul += perte.item()
        print(f"  époque {epoque+1} : perte {cumul/len(donnees):.4f} "
              f"en {time.time()-depart:.0f} s")
    modele.eval()
    return modele


@torch.no_grad()
def predire(modele, tokenizer, paires, taille_lot=128, longueur=256, colonne=1):
    sorties = []
    for i in range(0, len(paires), taille_lot):
        lot = paires[i:i + taille_lot]
        encode = tokenizer([a for a, _ in lot], [b for _, b in lot], padding=True,
                           truncation=True, max_length=longueur,
                           return_tensors="pt").to("cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            proba = torch.softmax(modele(**encode).logits.float(), dim=-1)
        sorties.append(proba[:, colonne].cpu().numpy() if colonne is not None
                       else proba.cpu().numpy())
    if not sorties:
        return np.zeros(0)
    return np.concatenate(sorties) if colonne is not None else np.vstack(sorties)

## Premier modèle, choisir les phrases qui justifient

Une phrase est un exemple positif si elle fait partie d'une justification du jeu
de données. Les négatifs viennent des articles cités sans preuve et des autres
phrases des articles qui en contiennent une, exactement comme dans le papier.

Les phrases justificatives sont rares, autour d'une sur huit, donc je repondère
pour que le modèle n'apprenne pas simplement à répondre non.

Je compare SciBERT, pré-entraîné sur des articles scientifiques, et
DeBERTa-v3-base qui est plus récent mais généraliste. Le choix se fait sur le
jeu de réglage.

In [ ]:
def phrases_et_etiquettes(affirmations):
    entrees, cibles = [], []
    for affirmation in affirmations:
        preuves = affirmation.get("evidence") or {}
        for article in affirmation.get("cited_doc_ids", []):
            article = str(article)
            if article not in articles:
                continue
            justificatives = {i for groupe in preuves.get(article, [])
                              for i in groupe["sentences"]}
            for i, phrase in enumerate(articles[article]["abstract"]):
                entrees.append((phrase, affirmation["claim"]))
                cibles.append(1 if i in justificatives else 0)
    return entrees, np.array(cibles)


phrases_entrainement, vraies_entrainement = phrases_et_etiquettes(entrainement)
phrases_reglage, vraies_reglage = phrases_et_etiquettes(reglage)
phrases_dev, vraies_dev = phrases_et_etiquettes(developpement)

part = 100 * vraies_entrainement.mean()
print(f"{len(phrases_entrainement)} phrases, dont {part:.1f} % justificatives")


def precision_rappel(notes, vraies, seuil):
    retenues = notes >= seuil
    justes = int((retenues & (vraies == 1)).sum())
    fausses = int((retenues & (vraies == 0)).sum())
    ratees = int((~retenues & (vraies == 1)).sum())
    p = justes / (justes + fausses) if justes + fausses else 0.0
    r = justes / (justes + ratees) if justes + ratees else 0.0
    return p, r, 2 * p * r / (p + r) if p + r else 0.0


poids = torch.tensor([1.0, float((vraies_entrainement == 0).sum() /
                                 max((vraies_entrainement == 1).sum(), 1))])
print(f"poids de la classe positive : {poids[1]:.2f}")

candidats = {}
for nom in ["allenai/scibert_scivocab_uncased", "microsoft/deberta-v3-base"]:
    print(f"\n{nom}")
    tokenizer = AutoTokenizer.from_pretrained(nom)
    modele = charger_modele(nom, nb_classes=2)
    modele = entrainer(modele, tokenizer, phrases_entrainement, vraies_entrainement,
                       poids_classes=poids)
    notes = predire(modele, tokenizer, phrases_reglage)
    seuil, f1 = max(((s, precision_rappel(notes, vraies_reglage, s)[2])
                     for s in np.arange(0.05, 0.96, 0.05)), key=lambda x: x[1])
    candidats[nom] = (modele, tokenizer, float(seuil), float(f1))
    print(f"  F1 sur le jeu de réglage : {100*f1:.1f} au seuil {seuil:.2f}")

choisi = max(candidats, key=lambda n: candidats[n][3])
selecteur, tokenizer_selecteur, seuil_phrases, _ = candidats[choisi]
print(f"\nretenu : {choisi}")

import gc
for nom in list(candidats):
    if nom != choisi:
        del candidats[nom]
gc.collect()
torch.cuda.empty_cache()

p, r, f1 = precision_rappel(predire(selecteur, tokenizer_selecteur, phrases_dev),
                            vraies_dev, seuil_phrases)
print(f"sur le développement avec les bons articles : "
      f"précision {100*p:.1f}, rappel {100*r:.1f}, F1 {100*f1:.1f}")
print("le papier annonce 72,1 pour RoBERTa-large et 74,4 pour SciBERT")

## Second modèle, décider l'étiquette

Il part d'un modèle déjà entraîné sur de l'inférence textuelle, dont les trois
sorties sont entailment, neutral et contradiction. Ça tombe bien : ce sont
exactement étayé, sans information et contredit. On garde donc sa tête au lieu
d'en repartir de zéro.

Point important, appris à mes dépens : il faut l'entraîner sur ce que le
sélecteur produit réellement, pas sur les justifications idéales. Sinon il
apprend un raccourci et s'effondre dès qu'on enchaîne les deux modèles.

Quand aucune phrase ne passe le seuil, on garde quand même la mieux notée. Sans
ça les articles sans preuve disparaissent de l'entraînement et le modèle
n'apprend jamais à dire non.

In [ ]:
nb_phrases_max = 3

def choisir_phrases(affirmation, article, seuil):
    phrases = articles[article]["abstract"]
    if not phrases:
        return []
    notes = predire(selecteur, tokenizer_selecteur, [(p, affirmation) for p in phrases])
    retenues = np.where(notes >= seuil)[0]
    if len(retenues) == 0:
        retenues = np.array([int(notes.argmax())])
    retenues = retenues[np.argsort(-notes[retenues])][:nb_phrases_max]
    return sorted(int(i) for i in retenues)


def exemples_etiquette(affirmations, seuil):
    entrees, etiquettes = [], []
    for affirmation in affirmations:
        preuves = affirmation.get("evidence") or {}
        for article in affirmation.get("cited_doc_ids", []):
            article = str(article)
            if article not in articles:
                continue
            indices = choisir_phrases(affirmation["claim"], article, seuil)
            if not indices:
                continue
            texte = " ".join(articles[article]["abstract"][i] for i in indices)
            entrees.append((texte, affirmation["claim"]))
            groupe = preuves.get(article)
            etiquettes.append(groupe[0]["label"] if groupe else "NOINFO")
    return entrees, etiquettes


nom_classifieur = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
tokenizer_classifieur = AutoTokenizer.from_pretrained(nom_classifieur)
classifieur = charger_modele(nom_classifieur)

noms_sorties = {int(k): v.lower() for k, v in classifieur.config.id2label.items()}
vers_sortie = {
    "SUPPORT":    next(i for i, v in noms_sorties.items() if v.startswith("entail")),
    "NOINFO":     next(i for i, v in noms_sorties.items() if v.startswith("neutral")),
    "CONTRADICT": next(i for i, v in noms_sorties.items() if v.startswith("contradic")),
}
vers_etiquette = {v: k for k, v in vers_sortie.items()}

entrees, etiquettes = exemples_etiquette(entrainement, seuil_phrases)
cibles = np.array([vers_sortie[e] for e in etiquettes])
print(f"{len(entrees)} exemples :", dict(Counter(etiquettes)))
assert etiquettes.count("NOINFO") >= 50, "trop peu d'exemples sans information"

classifieur = entrainer(classifieur, tokenizer_classifieur, entrees, cibles,
                        taille_lot=4, pas=1e-5, longueur=320)


def etiqueter(entrees):
    return predire(classifieur, tokenizer_classifieur, entrees,
                   taille_lot=16, longueur=320, colonne=None)


entrees_dev, etiquettes_dev = exemples_etiquette(developpement, seuil_phrases)
sorties = etiqueter(entrees_dev).argmax(1)
attendues = np.array([vers_sortie[e] for e in etiquettes_dev])
print(f"\nexactitude sur le développement : {100 * (sorties == attendues).mean():.1f}")
print("le papier annonce 75,7, mais avec les justifications idéales en entrée,")
print("donc dans des conditions plus faciles que celles-ci")
for etiquette, sortie in vers_sortie.items():
    lesquels = attendues == sortie
    if lesquels.sum():
        print(f"  {etiquette:<11} {int(lesquels.sum()):>4} cas, "
              f"{100 * (sorties[lesquels] == sortie).mean():.1f} % justes")

## La chaîne complète

Le seuil et le nombre d'articles examinés se règlent sur le jeu mis de côté, en
optimisant la métrique qui sera rapportée et pas une autre.

In [ ]:
def verifier(affirmations, trouves, seuil, nb_articles):
    resultats = []
    for affirmation in affirmations:
        preuve = {}
        for article in trouves[affirmation["id"]][:nb_articles]:
            indices = choisir_phrases(affirmation["claim"], article, seuil)
            if not indices:
                continue
            texte = " ".join(articles[article]["abstract"][i] for i in indices)
            sortie = etiqueter([(texte, affirmation["claim"])])[0].argmax()
            etiquette = vers_etiquette[int(sortie)]
            if etiquette != "NOINFO":
                preuve[str(article)] = {"label": etiquette, "sentences": indices}
        resultats.append({"id": affirmation["id"], "evidence": preuve})
    return resultats


def evaluer(resultats, fichier_or):
    with open("predictions.jsonl", "w") as f:
        for ligne in resultats:
            f.write(json.dumps(ligne) + "\n")
    sortie = subprocess.run(
        ["python", "pipeline.py", "--gold", f"../{fichier_or}",
         "--corpus", "../data/corpus.jsonl", "--prediction", "../predictions.jsonl",
         "--output", "../mesures.json"],
        cwd="evaluate", capture_output=True, text=True)
    if not os.path.exists("mesures.json"):
        print(sortie.stdout[-1000:], sortie.stderr[-1000:])
        raise RuntimeError("l'évaluateur a échoué")
    mesures = json.load(open("mesures.json"))
    os.remove("mesures.json")
    return mesures


meilleur = None
meilleure_valeur = 0.0
for seuil in (0.3, 0.5, 0.7, 0.85, 0.95):
    for nb_articles in (3, 5):
        mesures = evaluer(verifier(reglage, retrouves["reglage"], seuil, nb_articles),
                          "data/claims_reglage.jsonl")
        valeur = mesures["abstract_rationalized"]["f1"] * 100
        print(f"seuil {seuil:<5} {nb_articles} articles : {valeur:.1f}")
        if valeur > meilleure_valeur:
            meilleure_valeur = valeur
            meilleur = (seuil, nb_articles)

seuil_final, nb_articles_final = meilleur
print(f"\nretenu : seuil {seuil_final}, {nb_articles_final} articles")

## Le résultat

In [ ]:
predictions = verifier(developpement, retrouves["developpement"],
                       seuil_final, nb_articles_final)
mesures = evaluer(predictions, "data/claims_dev.jsonl")

phrases = mesures["sentence_label"]["f1"] * 100
articles_f1 = mesures["abstract_rationalized"]["f1"] * 100

reperes = [("zéro-shot FEVER, 2020", 28.4, 38.4),
           ("VeriSci", 42.6, 48.5),
           ("VeriSci, bons articles fournis", 60.6, 72.5),
           ("justifications fournies", 79.9, 83.0)]

print(f"{'':<34}{'phrases':>10}{'articles':>11}")
for nom, p, a in reperes:
    print(f"{nom:<34}{p:>10.1f}{a:>11.1f}")
print(f"{'ce système':<34}{phrases:>10.1f}{articles_f1:>11.1f}")
print()
for cle, nom in [("sentence_selection", "phrases retenues"),
                 ("sentence_label", "phrases retenues et étiquetées"),
                 ("abstract_label_only", "articles étiquetés"),
                 ("abstract_rationalized", "articles étiquetés et justifiés")]:
    m = mesures[cle]
    print(f"  {nom:<34} précision {m['precision']*100:>5.1f}  "
          f"rappel {m['recall']*100:>5.1f}  F1 {m['f1']*100:>5.1f}")

## Est-ce que l'écart avec VeriSci est réel

Le F1 se calcule sur l'ensemble des affirmations, il ne se moyenne pas une par
une. Je rééchantillonne donc les affirmations et je le recalcule à chaque tirage.

In [ ]:
def f1_articles(predictions, affirmations):
    justes = fausses = ratees = 0
    par_identifiant = {p["id"]: p for p in predictions}
    for affirmation in affirmations:
        attendu = affirmation.get("evidence") or {}
        obtenu = par_identifiant[affirmation["id"]]["evidence"]
        for article, prediction in obtenu.items():
            groupes = attendu.get(article)
            correct = False
            if groupes and groupes[0]["label"] == prediction["label"]:
                correct = any(set(g["sentences"]) <= set(prediction["sentences"])
                              for g in groupes)
            justes += correct
            fausses += not correct
        ratees += sum(1 for article in attendu if article not in obtenu)
    p = justes / (justes + fausses) if justes + fausses else 0.0
    r = justes / (justes + ratees) if justes + ratees else 0.0
    return 2 * p * r / (p + r) if p + r else 0.0


tirage = np.random.default_rng(0)
valeurs = []
for _ in range(2000):
    echantillon = [developpement[i] for i in tirage.choice(len(developpement),
                                                           len(developpement))]
    valeurs.append(f1_articles(predictions, echantillon) * 100)
bas, haut = np.percentile(valeurs, [2.5, 97.5])

print(f"F1 mesuré : {articles_f1:.1f}")
print(f"intervalle de confiance à 95 % : [{bas:.1f}, {haut:.1f}]")
if bas <= 48.5 <= haut:
    print("VeriSci est à 48,5, dans l'intervalle : les deux systèmes se valent")
else:
    print("VeriSci est à 48,5, hors de l'intervalle : l'écart est réel")

json.dump({"selecteur": choisi, "classifieur": nom_classifieur,
           "seuil": seuil_final, "articles": nb_articles_final,
           "mesures": mesures, "intervalle": [bas, haut]},
          open("resultats_verification.json", "w"), indent=2)

## Quelques exemples

Le chiffre ne dit pas à quoi ressemble une réponse. Voici ce que le système
produit réellement, avec la vérité en dessous.

In [ ]:
montres = 0
par_identifiant = {p["id"]: p for p in predictions}
for affirmation in developpement:
    prediction = par_identifiant[affirmation["id"]]
    if not prediction["evidence"] or montres >= 3:
        continue
    montres += 1
    print(affirmation["claim"])
    for article, decision in prediction["evidence"].items():
        attendu = (affirmation.get("evidence") or {}).get(article)
        juste = attendu and attendu[0]["label"] == decision["label"]
        print(f"\n  {decision['label']}{'  (juste)' if juste else '  (faux)'}")
        print(f"  {articles[article]['title']}")
        for i in decision["sentences"]:
            print(f"    {articles[article]['abstract'][i].strip()}")
        if attendu:
            attendues = sorted({s for g in attendu for s in g["sentences"]})
            print(f"  attendu : {attendu[0]['label']}, phrases {attendues}")
        else:
            print("  attendu : cet article n'est pas une preuve")
    print()